## This code reads 'seascape_stations_grazing.tsv' containing time stamps and coordinates of grazing experiment incubations done by Nicole Millete et al during SFER cruises, and extracts the 8-day and monthly seascapes for each incubation event

### Import Libraries & Prepare Station Stats

In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.spatial import KDTree
from scipy.stats import mode as sp_mode
import requests
import os
import warnings
warnings.filterwarnings('ignore')

# --- Load Data ---
# file_path = 'seascape_stations_grazing.tsv'
file_path = 'neslter_seascape_sites.tsv'
df = pd.read_csv(file_path, sep='\t')

# Sort the table based on the 'date' column in ascending order
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# --- Calculate mean latitude, longitude, and std for each station ---
station_stats = df.groupby('station_id').agg(
    MeanLatitude=('latitude', 'mean'),
    StdLatitude=('latitude', 'std'),
    MeanLongitude=('longitude', 'mean'),
    StdLongitude=('longitude', 'std')
).reset_index()

print("Station Stats:")
display(station_stats)

# --- Create 'events' table ---
# Merge original data with the corresponding mean latitude and longitude
events = df.merge(station_stats[['station_id', 'MeanLatitude', 'MeanLongitude']], on='station_id', how='left')

print("\nEvents Table:")
display(events.head())

Station Stats:


,station_id,MeanLatitude,StdLatitude,MeanLongitude,StdLongitude
0,L1,41.197700,0.003703,-70.882625,0.005196
1,L1.2,41.161800,NaN,-70.891300,NaN
2,L10,39.935060,0.009014,-70.887150,0.007880
3,L11,39.771550,0.006641,-70.885070,0.009703
4,L2,41.030887,0.003062,-70.883027,0.003556
5,L3,40.861120,0.003689,-70.885320,0.002834
6,L4,40.696570,0.003471,-70.884315,0.005983
7,L6,40.361733,0.005712,-70.886120,0.008791
8,L7,40.226500,NaN,-70.883200,NaN
9,L8,40.141011,0.011330,-70.831306,0.057444



Events Table:


,station_id,date,latitude,longitude,MeanLatitude,MeanLongitude
0,L1,2018-01-31,41.1964,-70.8777,41.197700,-70.882625
1,L3,2018-02-01,40.8559,-70.8844,40.861120,-70.885320
2,L4,2018-02-01,40.6928,-70.8842,40.696570,-70.884315
3,L8,2018-02-02,40.1200,-70.8063,40.141011,-70.831306
4,L11,2018-02-03,39.7631,-70.8898,39.771550,-70.885070


### Define Domain Limits & Download Seascape Data from ERDDAP 

In [10]:
# --- Choose Dataset Type ---
# Set this to either 'SEASCAPE_8DAY' or 'SEASCAPE_MONTH'
dataset_type = 'SEASCAPE_8DAY' 

# --- Study area limits ---
# n_lat, s_lat = 29.0, 23.0
# e_lon, w_lon = -79.5, -86.0
n_lat, s_lat = 42.0, 38.0
e_lon, w_lon = -68.0, -72.0
start_date = '2018-01-15T12:00:00Z'
end_date = '2025-09-15T12:00:00Z'

# Construct the URL based on the chosen dataset type
url = (f"https://cwcgom.aoml.noaa.gov/thredds/ncss/{dataset_type}/SEASCAPES.nc?"
       f"var=CLASS&var=P&north={n_lat}&west={w_lon}&east={e_lon}&south={s_lat}&"
       f"disableProjSubset=on&horizStride=1&time_start={start_date}&time_end={end_date}&"
       f"timeStride=1&addLatLon=true&accept=netcdf")

# Dynamically name the file so 8-DAY and MONTHLY don't overwrite each other
filename = f'{dataset_type}_data_grazing.nc'

# Download the file if it doesn't already exist
if not os.path.exists(filename):
    print(f"Downloading {dataset_type} data from: \n{url}\n")
    r = requests.get(url, timeout=1200)
    if r.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(r.content)
        print("Download complete.")
    else:
        print(f"Failed to download: HTTP {r.status_code}")
else:
    print(f"File '{filename}' already exists. Skipping download.")

# Open the NetCDF file
ds = xr.open_dataset(filename, engine='netcdf4')
print(ds)

https://cwcgom.aoml.noaa.gov/thredds/ncss/SEASCAPE_8DAY/SEASCAPES.nc?var=CLASS&var=P&north=42.0&west=-72.0&east=-68.0&south=38.0&disableProjSubset=on&horizStride=1&time_start=2018-01-15T12:00:00Z&time_end=2025-09-15T12:00:00Z&timeStride=1&addLatLon=true&accept=netcdf

Download complete.
<xarray.Dataset> Size: 18MB
Dimensions:  (time: 351, lat: 81, lon: 81)
Coordinates:
  * time     (time) datetime64[ns] 3kB 2018-01-17T12:00:00 ... 2025-09-14T12:...
  * lat      (lat) float32 324B 38.03 38.08 38.12 38.17 ... 41.92 41.97 42.03
  * lon      (lon) float32 324B -71.97 -71.93 -71.88 ... -68.07 -68.03 -67.97
Data variables:
    CLASS    (time, lat, lon) float32 9MB ...
    P        (time, lat, lon) float32 9MB ...
Attributes: (12/32)
    Conventions:                CF-1.6,ACDD-1.3
    title:                      Global Seascapes 8DAY: Prototype
    institution:                NOAA CoastWatch, OSU, USF, NASA, UAF, IOOS, NMS
    summary:                    Biogeographic framework. Space and tim

### Map Setup & Spatial KDTree Initialization
#### This cell builds a custom dynamic colormap and sets up a KDTree for rapid spatial radius searching (converting Matlab's 3-pixel radius search at 4km resolution into degrees).

In [6]:
# --- Prepare Spatial Search (KDTree) ---
n_pix = 3
pix_res = 24  # 24 indicates 4-km seascape resolution
search_radius_deg = n_pix / pix_res

# Flatten the lat/lon grid for rapid nearest-neighbor searches
lon_grid, lat_grid = np.meshgrid(ds.lon.values, ds.lat.values)
grid_points = np.vstack([lat_grid.ravel(), lon_grid.ravel()]).T
kdtree = KDTree(grid_points)

# --- Dynamic Color Palette ---
# Finds all unique classes to ensure colors never repeat across maps
all_values = ds['CLASS'].values
global_classes = np.unique(all_values[~np.isnan(all_values)]).astype(int)
num_global_classes = len(global_classes)

if num_global_classes <= 20:
    colors = plt.get_cmap('tab20b')(np.linspace(0, 1, num_global_classes))
else:
    combined_palette = np.vstack([
        plt.get_cmap('tab20b')(np.linspace(0, 1, 20)),
        plt.get_cmap('tab20c')(np.linspace(0, 1, 20))
    ])
    colors = combined_palette[:num_global_classes]

common_cmap = ListedColormap(colors)
common_bounds = np.append(global_classes, global_classes[-1] + 1) - 0.5
common_norm = BoundaryNorm(common_bounds, ncolors=common_cmap.N)

### Process 8-Day or Monthly Windows, Export Maps, and Extract Pixels
#### This loop iterates through the 8-day or monthly time bins, isolates matching sample events, exports a localized geographic map to a subfolder, and calculates the modal seascape probability around the stations.

In [11]:
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- Define Fixed Universal Color Palette ---
# NOAA Seascapes have 33 total global classes. By fixing the palette to 1-33, 
# Class X will ALWAYS have the exact same color across 8-day, monthly, or any time step.
fixed_classes = np.arange(1, 34)
combined_palette = np.vstack([
    plt.get_cmap('tab20b')(np.linspace(0, 1, 20)),
    plt.get_cmap('tab20c')(np.linspace(0, 1, 20))
])
fixed_colors = combined_palette[:len(fixed_classes)]
fixed_cmap = ListedColormap(fixed_colors)
fixed_bounds = np.append(fixed_classes, fixed_classes[-1] + 1) - 0.5
fixed_norm = BoundaryNorm(fixed_bounds, ncolors=fixed_cmap.N)

# --- Process Data and Generate Maps ---
conc_class_per_site = []
times = ds.time.values

# Create directory to store the output PNGs
os.makedirs('grazing_seascape_maps', exist_ok=True)

# Loop through each time step
for j in range(len(times)):
    start_t = pd.to_datetime(times[j])
    
    # --- Define Time Window Based on Dataset Type ---
    if dataset_type == 'SEASCAPE_8DAY':
        if j == len(times) - 1:
            break  # Skip the very last index for 8-day because we need j+1 to define the window end
        end_t = pd.to_datetime(times[j+1])
        wk_idx = events[(events['date'] >= start_t) & (events['date'] < end_t)]
        time_str = start_t.strftime('%Y-%m-%d')
    else:  # SEASCAPE_MONTH
        # For monthly, match the year and month directly
        wk_idx = events[(events['date'].dt.year == start_t.year) & (events['date'].dt.month == start_t.month)]
        time_str = start_t.strftime('%Y-%m')
    
    if not wk_idx.empty:
        print(f"Processing window: {time_str} with {len(wk_idx)} events")
        
        sel_seas = ds.isel(time=j)['CLASS'].values
        sel_prob = ds.isel(time=j)['P'].values
        
        # --- 1. Map Seascapes ---
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
        ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())
        
        # Applied the fixed_cmap and fixed_norm here
        mesh = ax.pcolormesh(ds.lon.values, ds.lat.values, sel_seas, cmap=fixed_cmap, norm=fixed_norm, transform=ccrs.PlateCarree(), zorder=1)
        
        ax.add_feature(cfeature.LAND, facecolor='0.7', edgecolor='none', zorder=2)
        ax.add_feature(cfeature.COASTLINE, zorder=3)
        ax.gridlines(draw_labels=True, linewidth=2, color='gray', alpha=0.5, linestyle='--')
        
        # Plot corresponding station events
        ax.scatter(wk_idx['MeanLongitude'], wk_idx['MeanLatitude'], c='white', edgecolors='black', marker='o', s=50, transform=ccrs.PlateCarree(), zorder=4)
        ax.scatter(wk_idx['MeanLongitude'], wk_idx['MeanLatitude'], c='black', marker='*', s=50, transform=ccrs.PlateCarree(), zorder=5)
        
        # Dynamically set colorbar ticks to only show classes present in this specific slice
        slice_classes = np.unique(sel_seas[~np.isnan(sel_seas)]).astype(int)
        cbar = plt.colorbar(mesh, ax=ax, ticks=slice_classes, pad=0.04)
        cbar.set_label('Seascape Class')
        plt.title(time_str, weight='bold', fontsize=14)
        
        # Save map and close figure
        plt.savefig(f"grazing_seascape_maps/{time_str}.png", dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        # --- 2. Extract Surrounding Seascape Classes ---
        station_points = wk_idx[['MeanLatitude', 'MeanLongitude']].values
        nearby_indices_list = kdtree.query_ball_point(station_points, r=search_radius_deg)
        
        for idx, (original_idx, station) in enumerate(wk_idx.iterrows()):
            flat_indices = nearby_indices_list[idx]
            
            if not flat_indices:
                seas_mode = np.nan
                seas_prob_mean = np.nan
            else:
                # Find valid pixels within radius
                rows, cols = np.unravel_index(flat_indices, sel_seas.shape)
                nearby_classes = sel_seas[rows, cols]
                nearby_probs = sel_prob[rows, cols]
                
                valid_mask = ~np.isnan(nearby_classes)
                valid_classes = nearby_classes[valid_mask]
                
                if valid_classes.size > 0:
                    # Calculate seascape mode
                    seas_mode = sp_mode(valid_classes, keepdims=False).mode
                    # Calculate mean probability of modal seascape
                    valid_probs = nearby_probs[valid_mask]
                    probs_of_modal = valid_probs[valid_classes == seas_mode]
                    seas_prob_mean = np.mean(probs_of_modal)
                else:
                    seas_mode = np.nan
                    seas_prob_mean = np.nan
            
            class_per_site = {
                'Date': station['date'],
                'Latitude': station['MeanLatitude'],
                'Longitude': station['MeanLongitude'],
                'Seascape': seas_mode,
                'Probability': seas_prob_mean
            }
            conc_class_per_site.append(class_per_site)

Processing window: 2018-01-25 with 4 events
Processing window: 2018-02-02 with 1 events
Processing window: 2018-07-12 with 1 events
Processing window: 2018-07-20 with 5 events
Processing window: 2019-01-25 with 3 events
Processing window: 2019-02-02 with 4 events
Processing window: 2020-01-25 with 2 events
Processing window: 2020-02-02 with 5 events
Processing window: 2020-07-19 with 4 events
Processing window: 2020-10-07 with 5 events
Processing window: 2020-10-15 with 2 events
Processing window: 2021-02-02 with 6 events
Processing window: 2021-07-12 with 6 events
Processing window: 2022-02-10 with 4 events
Processing window: 2022-02-18 with 3 events
Processing window: 2022-04-07 with 5 events
Processing window: 2022-04-23 with 3 events
Processing window: 2022-07-28 with 7 events
Processing window: 2023-01-09 with 6 events
Processing window: 2023-08-05 with 6 events
Processing window: 2023-10-08 with 9 events
Processing window: 2024-02-02 with 2 events
Processing window: 2024-02-10 wi

### Merge Results and Export to CSV

In [12]:
# --- Convert the extracted array into a DataFrame ---
T1 = pd.DataFrame(conc_class_per_site)

# --- Prepare the 'events' table for merging ---
# Rename the columns in 'events' to match the key names for joining
T2 = events.rename(columns={
    'date': 'Date', 
    'MeanLatitude': 'Latitude', 
    'MeanLongitude': 'Longitude'
})

# --- Merge the two tables ---
# Inner join keeps only the rows where the key values exist in BOTH tables
merged_table = pd.merge(T1, T2, on=['Date', 'Latitude', 'Longitude'], how='inner')

print("Merge successful. Displaying the first 5 rows:")
display(merged_table.head())

# Write the table to a CSV file explicitly naming the dataset type
output_file = f'seascape_class_per_event_{dataset_type}.csv'
merged_table.to_csv(output_file, index=False)
print(f"\nSaved successfully to {output_file}")

Merge successful. Displaying the first 5 rows:


,Date,Latitude,Longitude,Seascape,Probability,station_id,latitude,longitude
0,2018-01-31,41.197700,-70.882625,19.0,0.779808,L1,41.1964,-70.8777
1,2018-02-01,40.861120,-70.885320,14.0,0.980766,L3,40.8559,-70.8844
2,2018-02-01,40.696570,-70.884315,14.0,0.990394,L4,40.6928,-70.8842
3,2018-02-02,40.141011,-70.831306,14.0,0.996837,L8,40.1200,-70.8063
4,2018-02-03,39.771550,-70.885070,NaN,NaN,L11,39.7631,-70.8898



Saved successfully to seascape_class_per_event_SEASCAPE_8DAY.csv


### Merge 8-Day and Monthly Data tables into a single one

In [13]:
import pandas as pd

# --- 1. Load the two CSV files ---
try:
    df_8day = pd.read_csv('seascape_class_per_event_SEASCAPE_8DAY.csv')
    df_month = pd.read_csv('seascape_class_per_event_SEASCAPE_MONTH.csv')
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    print("Make sure you have run the extraction loop for BOTH dataset types before running this cell.")
    raise

# --- 2. Rename columns to distinguish between the datasets ---
df_8day = df_8day.rename(columns={
    'Seascape': 'Seascape_8day',
    'Probability': 'Probability_8day'
})

df_month = df_month.rename(columns={
    'Seascape': 'Seascape_monthly',
    'Probability': 'Probability_monthly'
})

# --- 3. Merge the datasets ---
# We merge on the core identifying columns. 
# Using 'outer' ensures no records are lost if an event was only extracted in one of the datasets.
merge_cols = ['station_id', 'Date', 'Latitude', 'Longitude']

merged_df = pd.merge(
    df_8day[merge_cols + ['Seascape_8day', 'Probability_8day']], 
    df_month[merge_cols + ['Seascape_monthly', 'Probability_monthly']], 
    on=merge_cols, 
    how='outer'
)

# --- 4. Extract Date Components ---
# Ensure the 'Date' column is a datetime object so we can extract Year, Month, and Day
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

merged_df['Year'] = merged_df['Date'].dt.year
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Day'] = merged_df['Date'].dt.day

# Convert Date back to a clean string format (YYYY-MM-DD)
merged_df['Date'] = merged_df['Date'].dt.strftime('%Y-%m-%d')

# --- 5. Reorder Columns ---
final_columns = [
    'station_id', 'Year', 'Month', 'Day', 'Date', 
    'Latitude', 'Longitude', 
    'Seascape_8day', 'Probability_8day', 
    'Seascape_monthly', 'Probability_monthly'
]
merged_df = merged_df[final_columns]

# --- 6. Display and Save ---
print("Merged Table Preview:")
display(merged_df.head(10))

output_merged_file = 'seascape_class_per_event_COMBINED.csv'
merged_df.to_csv(output_merged_file, index=False)
print(f"\nSaved combined dataset to: {output_merged_file}")

Merged Table Preview:


,station_id,Year,Month,Day,Date,Latitude,Longitude,Seascape_8day,Probability_8day,Seascape_monthly,Probability_monthly
0,L1,2018,1,31,2018-01-31,41.1977,-70.882625,19.0,0.779808,19.0,0.711640
1,L1,2018,7,20,2018-07-20,41.1977,-70.882625,19.0,0.738914,19.0,0.672621
2,L1,2019,2,1,2019-02-01,41.1977,-70.882625,14.0,0.773150,19.0,0.799401
3,L1,2020,2,1,2020-02-01,41.1977,-70.882625,14.0,0.995886,14.0,0.839334
4,L1,2020,7,25,2020-07-25,41.1977,-70.882625,20.0,0.785290,20.0,0.536287
5,L1,2020,10,13,2020-10-13,41.1977,-70.882625,21.0,0.990226,21.0,0.988940
6,L1,2021,2,3,2021-02-03,41.1977,-70.882625,14.0,0.791864,14.0,0.808099
7,L1,2021,7,16,2021-07-16,41.1977,-70.882625,21.0,0.738483,21.0,0.807102
8,L1,2022,2,16,2022-02-16,41.1977,-70.882625,14.0,0.998669,14.0,0.980259
9,L1,2022,7,29,2022-07-29,41.1977,-70.882625,21.0,0.729197,21.0,0.805672



Saved combined dataset to: seascape_class_per_event_COMBINED.csv
